# 01 - Ingestão dos Dados | Camada Bronze

Este notebook realiza a ingestão dos microdados da PNAD Contínua utilizados no projeto, correspondentes aos períodos de 2022 e 2024.

A Camada Bronze constitui a etapa inicial da arquitetura de dados adotada no projeto. Seu objetivo é armazenar os dados provenientes da fonte original preservando sua estrutura bruta, de modo que as etapas posteriores de tratamento e transformação possam ser realizadas sem modificar os arquivos de origem.

Os microdados da PNAD Contínua são disponibilizados em arquivos de largura fixa. Nesta etapa, os arquivos são obtidos, extraídos e armazenados no ambiente Databricks, mantendo-se os registros originais para posterior processamento.

## Objetivos desta etapa

- realizar a ingestão dos microdados da PNAD Contínua de 2022 e 2024;
- armazenar os arquivos originais na Camada Bronze;
- preservar a estrutura dos dados provenientes da fonte;
- verificar a integridade dos arquivos após a ingestão;
- disponibilizar os dados brutos para as etapas posteriores do pipeline.

Ao final desta etapa, os arquivos permanecem armazenados sem transformações analíticas, constituindo a fonte de dados para a construção da Camada Silver.

In [0]:
# Caminhos dos arquivos brutos da PNAD Contínua

path_2022 = "/Volumes/workspace/bronze/pnad_raw/2022/"
path_2024 = "/Volumes/workspace/bronze/pnad_raw/2024/"

print("Arquivos de 2022:")
display(dbutils.fs.ls(path_2022))

print("Arquivos de 2024:")
display(dbutils.fs.ls(path_2024))

In [0]:
import zipfile

zip_2022 = "/Volumes/workspace/bronze/pnad_raw/2022/PNADC_2022_trimestre4_20251010.zip"
zip_2024 = "/Volumes/workspace/bronze/pnad_raw/2024/PNADC_2024_trimestre3_20260904.zip"

print("Conteúdo do ZIP de 2022:")
with zipfile.ZipFile(zip_2022, "r") as arquivo:
    for nome in arquivo.namelist():
        print(nome)

print("\nConteúdo do ZIP de 2024:")
with zipfile.ZipFile(zip_2024, "r") as arquivo:
    for nome in arquivo.namelist():
        print(nome)

In [0]:
import zipfile
import os

destino_2022 = "/Volumes/workspace/bronze/pnad_raw/2022/extracted"
destino_2024 = "/Volumes/workspace/bronze/pnad_raw/2024/extracted"

os.makedirs(destino_2022, exist_ok=True)
os.makedirs(destino_2024, exist_ok=True)

with zipfile.ZipFile(zip_2022, "r") as arquivo:
    arquivo.extractall(destino_2022)

with zipfile.ZipFile(zip_2024, "r") as arquivo:
    arquivo.extractall(destino_2024)

print("Extração concluída.")

In [0]:
print("Arquivo extraído de 2022:")
display(dbutils.fs.ls(destino_2022))

print("Arquivo extraído de 2024:")
display(dbutils.fs.ls(destino_2024))


In [0]:
arquivo_2022 = "/Volumes/workspace/bronze/pnad_raw/2022/extracted/PNADC_2022_trimestre4.txt"

df_raw_2022 = spark.read.text(arquivo_2022)

display(df_raw_2022.limit(5))

In [0]:
from pyspark.sql.functions import col, substring

df_teste_2022 = (
    df_raw_2022
    .select(
        substring(col("value"), 1, 4).alias("ano"),
        substring(col("value"), 5, 1).alias("trimestre"),
        substring(col("value"), 6, 2).alias("uf"),
        substring(col("value"), 95, 1).alias("sexo"),
        substring(col("value"), 104, 3).alias("idade"),
        substring(col("value"), 107, 1).alias("cor_raca")
    )
)

display(df_teste_2022.limit(10))

In [0]:
from pyspark.sql.functions import col, substring

df_plataformas_2022 = (
    df_raw_2022
    .select(
        substring(col("value"), 1, 4).alias("ano"),
        substring(col("value"), 5, 1).alias("trimestre"),
        substring(col("value"), 6, 2).alias("uf"),

        substring(col("value"), 1080, 1).alias("app_taxi"),
        substring(col("value"), 1081, 1).alias("app_transporte"),
        substring(col("value"), 1082, 1).alias("app_entrega"),
        substring(col("value"), 1083, 1).alias("app_servicos")
    )
)

display(
    df_plataformas_2022
    .filter(col("app_entrega").isNotNull())
    .filter(col("app_entrega") != "")
    .limit(20)
)

In [0]:
from pyspark.sql.functions import col, trim, count

df_valores_app_entrega = (
    df_plataformas_2022
    .select(trim(col("app_entrega")).alias("app_entrega"))
    .groupBy("app_entrega")
    .agg(count("*").alias("quantidade"))
    .orderBy("app_entrega")
)

display(df_valores_app_entrega)

In [0]:
from pyspark.sql.functions import col, substring, trim, count

df_validacao_s140093 = (
    df_raw_2022
    .select(
        substring(col("value"), 1083, 1).alias("S140093")
    )
    .select(trim(col("S140093")).alias("S140093"))
    .groupBy("S140093")
    .agg(count("*").alias("quantidade"))
    .orderBy("S140093")
)

display(df_validacao_s140093)

In [0]:
arquivo_2024 = "/Volumes/workspace/bronze/pnad_raw/2024/extracted/PNADC_2024_trimestre3.txt"

df_raw_2024 = spark.read.text(arquivo_2024)

print("Total de registros em 2024:")
print(df_raw_2024.count())

In [0]:
from pyspark.sql.functions import col, substring, trim, count

df_validacao_s140093_2024 = (
    df_raw_2024
    .select(
        substring(col("value"), 684, 1).alias("S140093")
    )
    .select(trim(col("S140093")).alias("S140093"))
    .groupBy("S140093")
    .agg(count("*").alias("quantidade"))
    .orderBy("S140093")
)

display(df_validacao_s140093_2024)

In [0]:
from pyspark.sql.functions import col, substring, trim

# Leitura do peso amostral V1028
# Posição inicial = 50 | Tamanho = 15

df_peso_2022 = (
    df_raw_2022
    .select(
        trim(substring(col("value"), 50, 15)).alias("V1028")
    )
)

df_peso_2024 = (
    df_raw_2024
    .select(
        trim(substring(col("value"), 50, 15)).alias("V1028")
    )
)

print("2022:")
display(df_peso_2022.limit(10))

print("2024:")
display(df_peso_2024.limit(10))

In [0]:
from pyspark.sql.functions import col, trim, substring, min, max, avg

df_peso_validacao_2022 = (
    df_raw_2022
    .select(
        trim(substring(col("value"), 50, 15)).alias("V1028_texto")
    )
    .withColumn(
        "V1028",
        col("V1028_texto").cast("double")
    )
)

df_peso_validacao_2024 = (
    df_raw_2024
    .select(
        trim(substring(col("value"), 50, 15)).alias("V1028_texto")
    )
    .withColumn(
        "V1028",
        col("V1028_texto").cast("double")
    )
)

print("Resumo dos pesos - 2022:")
display(
    df_peso_validacao_2022.select(
        min("V1028").alias("peso_minimo"),
        max("V1028").alias("peso_maximo"),
        avg("V1028").alias("peso_medio")
    )
)

print("Resumo dos pesos - 2024:")
display(
    df_peso_validacao_2024.select(
        min("V1028").alias("peso_minimo"),
        max("V1028").alias("peso_maximo"),
        avg("V1028").alias("peso_medio")
    )
)

In [0]:
from pyspark.sql.functions import col, substring, trim, sum as spark_sum, count

# 2022
df_entregadores_2022 = (
    df_raw_2022
    .select(
        trim(substring(col("value"), 1083, 1)).alias("S140093"),
        trim(substring(col("value"), 50, 15)).cast("double").alias("V1028")
    )
    .filter(col("S140093") == "1")
)

# 2024
df_entregadores_2024 = (
    df_raw_2024
    .select(
        trim(substring(col("value"), 684, 1)).alias("S140093"),
        trim(substring(col("value"), 50, 15)).cast("double").alias("V1028")
    )
    .filter(col("S140093") == "1")
)

print("Entregadores - 2022:")
display(
    df_entregadores_2022.agg(
        count("*").alias("registros_amostrais"),
        spark_sum("V1028").alias("estimativa_ponderada")
    )
)

print("Entregadores - 2024:")
display(
    df_entregadores_2024.agg(
        count("*").alias("registros_amostrais"),
        spark_sum("V1028").alias("estimativa_ponderada")
    )
)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, when

print("Validação - 2022:")
display(
    df_entregadores_2022.agg(
        count("*").alias("total_entregadores_amostra"),
        spark_sum(
            when(col("V1028").isNull(), 1).otherwise(0)
        ).alias("pesos_nulos"),
        spark_sum(
            when(col("V1028").isNotNull(), 1).otherwise(0)
        ).alias("pesos_validos")
    )
)

print("Validação - 2024:")
display(
    df_entregadores_2024.agg(
        count("*").alias("total_entregadores_amostra"),
        spark_sum(
            when(col("V1028").isNull(), 1).otherwise(0)
        ).alias("pesos_nulos"),
        spark_sum(
            when(col("V1028").isNotNull(), 1).otherwise(0)
        ).alias("pesos_validos")
    )
)

In [0]:
from pyspark.sql.functions import col, substring, trim

# ============================================================
# VARIÁVEIS NECESSÁRIAS PARA RECONSTRUIR SD14001
# Regra oficial: 4º tri/2022 e 3º tri/2024
# ============================================================

df_base_2022 = (
    df_raw_2022
    .select(
        trim(substring(col("value"), 1081, 1)).alias("S140091"),
        trim(substring(col("value"), 1082, 1)).alias("S140092"),
        trim(substring(col("value"), 1083, 1)).alias("S140093"),
        trim(substring(col("value"), 1084, 1)).alias("S140094"),
        trim(substring(col("value"), 156, 1)).alias("V4012"),
        trim(substring(col("value"), 157, 1)).alias("V40121"),
        trim(substring(col("value"), 158, 5)).alias("V4013"),
        trim(substring(col("value"), 50, 15)).alias("V1028")
    )
)

df_base_2024 = (
    df_raw_2024
    .select(
        trim(substring(col("value"), 682, 1)).alias("S140091"),
        trim(substring(col("value"), 683, 1)).alias("S140092"),
        trim(substring(col("value"), 684, 1)).alias("S140093"),
        trim(substring(col("value"), 685, 1)).alias("S140094"),
        trim(substring(col("value"), 156, 1)).alias("V4012"),
        trim(substring(col("value"), 157, 1)).alias("V40121"),
        trim(substring(col("value"), 158, 5)).alias("V4013"),
        trim(substring(col("value"), 50, 15)).alias("V1028")
    )
)

print("Base 2022:")
display(df_base_2022.limit(10))

print("Base 2024:")
display(df_base_2024.limit(10))

In [0]:
from pyspark.sql.functions import col, when

# Atividades previstas na definição oficial de SD14001
atividades_plataforma = [
    "49030", "49040", "52020", "53002",
    "48020", "48030", "48041", "48042", "48050", "48060",
    "48071", "48072", "48073", "48074", "48075", "48076",
    "48077", "48078", "48079", "48080", "48090", "48100",
    "56011", "56012", "56020"
]

def criar_sd14001(df):

    # Parte da regra relacionada a S140093
    cond_s140093 = (
        (col("S140093") == "1") &
        (
            col("V4013").isin(atividades_plataforma) |
            (
                (col("V4012") == "5") |
                (col("V4012") == "6") |
                (
                    (col("V4012") == "7") &
                    (col("V40121") == "1")
                )
            )
        )
    )

    # Regra oficial para SD14001 = 1 (Sim)
    cond_sim = (
        (col("S140091") == "1") |
        (col("S140092") == "1") |
        cond_s140093 |
        (col("S140094") == "1")
    )

    # Regra oficial para SD14001 = 2 (Não)
    cond_nao = (
        (col("S140091") == "2") &
        (col("S140092") == "2") &
        (
            (col("S140093") == "2") |
            (
                (col("S140093") == "1") &
                col("V4013").isin(atividades_plataforma) &
                (
                    (col("V4012") == "1") |
                    (col("V4012") == "2") |
                    (col("V4012") == "3") |
                    (
                        (col("V4012") == "7") &
                        (
                            (col("V40121") == "2") |
                            (col("V40121") == "3")
                        )
                    )
                )
            )
        ) &
        (col("S140094") == "2")
    )

    return (
        df.withColumn(
            "SD14001",
            when(cond_sim, 1)
            .when(cond_nao, 2)
        )
    )


df_sd14001_2022 = criar_sd14001(df_base_2022)
df_sd14001_2024 = criar_sd14001(df_base_2024)

print("Distribuição de SD14001 - 2022:")
display(
    df_sd14001_2022
    .groupBy("SD14001")
    .count()
    .orderBy("SD14001")
)

print("Distribuição de SD14001 - 2024:")
display(
    df_sd14001_2024
    .groupBy("SD14001")
    .count()
    .orderBy("SD14001")
)

In [0]:
from pyspark.sql.functions import col, trim, count

def diagnostico_variaveis(df, ano):
    print(f"\n===== {ano} =====")

    for variavel in ["S140091", "S140092", "S140093", "S140094", "V4012", "V40121"]:
        print(f"\n{variavel}:")
        display(
            df
            .select(trim(col(variavel)).alias(variavel))
            .groupBy(variavel)
            .agg(count("*").alias("quantidade"))
            .orderBy(variavel)
        )

diagnostico_variaveis(df_base_2022, 2022)
diagnostico_variaveis(df_base_2024, 2024)

In [0]:
from pyspark.sql.functions import col, substring, trim

# Correção da extração de 2024
# S140094 está na posição 686, e não 685.
# A posição 685 corresponde a S140093A.

df_base_2024 = (
    df_raw_2024
    .select(
        trim(substring(col("value"), 682, 1)).alias("S140091"),
        trim(substring(col("value"), 683, 1)).alias("S140092"),
        trim(substring(col("value"), 684, 1)).alias("S140093"),
        trim(substring(col("value"), 685, 1)).alias("S140093A"),
        trim(substring(col("value"), 686, 1)).alias("S140094"),
        trim(substring(col("value"), 156, 1)).alias("V4012"),
        trim(substring(col("value"), 157, 1)).alias("V40121"),
        trim(substring(col("value"), 158, 5)).alias("V4013"),
        trim(substring(col("value"), 50, 15)).alias("V1028")
    )
)

print("Validação S140093A - 2024:")
display(
    df_base_2024
    .groupBy("S140093A")
    .count()
    .orderBy("S140093A")
)

print("Validação S140094 - 2024:")
display(
    df_base_2024
    .groupBy("S140094")
    .count()
    .orderBy("S140094")
)

In [0]:
from pyspark.sql.functions import col, when

# Atividades previstas na definição do IBGE

atividades_grupo_1 = [
    "49030", "49040", "52020", "53002"
]

atividades_grupo_2 = [
    "48020", "48030", "48041", "48042", "48050", "48060",
    "48071", "48072", "48073", "48074", "48075", "48076",
    "48077", "48078", "48079", "48080", "48090", "48100",
    "56011", "56012", "56020"
]


def criar_sd14001(df):

    # ---------------------------
    # SD14001 = 1 (SIM)
    # ---------------------------

    cond_entrega_grupo_1 = (
        (col("S140093") == "1") &
        col("V4013").isin(atividades_grupo_1)
    )

    cond_entrega_grupo_2 = (
        (col("S140093") == "1") &
        col("V4013").isin(atividades_grupo_2) &
        (
            col("V4012").isin(["5", "6"]) |
            (
                (col("V4012") == "7") &
                (col("V40121") == "1")
            )
        )
    )

    cond_sim = (
        (col("S140091") == "1") |
        (col("S140092") == "1") |
        cond_entrega_grupo_1 |
        cond_entrega_grupo_2 |
        (col("S140094") == "1")
    )

    # ---------------------------
    # SD14001 = 2 (NÃO)
    # ---------------------------

    cond_entrega_nao = (
        (col("S140093") == "2") |
        (
            (col("S140093") == "1") &
            col("V4013").isin(atividades_grupo_2) &
            (
                col("V4012").isin(["1", "3"]) |
                col("V40121").isin(["2", "3"])
            )
        )
    )

    cond_nao = (
        (col("S140091") == "2") &
        (col("S140092") == "2") &
        cond_entrega_nao &
        (col("S140094") == "2")
    )

    return (
        df.withColumn(
            "SD14001",
            when(cond_sim, 1)
            .when(cond_nao, 2)
        )
    )


df_sd14001_2022 = criar_sd14001(df_base_2022)
df_sd14001_2024 = criar_sd14001(df_base_2024)

print("Distribuição SD14001 - 2022:")
display(
    df_sd14001_2022
    .groupBy("SD14001")
    .count()
    .orderBy("SD14001")
)

print("Distribuição SD14001 - 2024:")
display(
    df_sd14001_2024
    .groupBy("SD14001")
    .count()
    .orderBy("SD14001")
)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count

def estimar_plataformizados(df):
    return (
        df
        .filter(col("SD14001") == 1)
        .select(
            col("V1028").cast("double").alias("peso")
        )
        .agg(
            count("*").alias("registros_amostrais"),
            spark_sum("peso").alias("estimativa_ponderada")
        )
    )

print("Estimativa de trabalhadores plataformizados - 2022:")
display(estimar_plataformizados(df_sd14001_2022))

print("Estimativa de trabalhadores plataformizados - 2024:")
display(estimar_plataformizados(df_sd14001_2024))

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count

# Entregadores por aplicativo - resposta "Sim" em S140093

df_entregadores_2022 = (
    df_base_2022
    .filter(col("S140093") == "1")
)

df_entregadores_2024 = (
    df_base_2024
    .filter(col("S140093") == "1")
)

print("Entregadores por aplicativo - 2022:")
display(
    df_entregadores_2022
    .select(col("V1028").cast("double").alias("peso"))
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso").alias("estimativa_ponderada")
    )
)

print("Entregadores por aplicativo - 2024:")
display(
    df_entregadores_2024
    .select(col("V1028").cast("double").alias("peso"))
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso").alias("estimativa_ponderada")
    )
)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum

def cruzar_entrega_plataformizacao(df):
    return (
        df
        .filter(col("S140093") == "1")
        .groupBy("SD14001")
        .agg(
            count("*").alias("registros_amostrais"),
            spark_sum(col("V1028").cast("double")).alias("estimativa_ponderada")
        )
        .orderBy("SD14001")
    )

print("Entregadores segundo SD14001 - 2022:")
display(cruzar_entrega_plataformizacao(df_sd14001_2022))

print("Entregadores segundo SD14001 - 2024:")
display(cruzar_entrega_plataformizacao(df_sd14001_2024))